In [19]:
from google.colab import files

uploaded = files.upload()

Saving Resume_data.zip to Resume_data.zip


In [20]:
print(uploaded.keys())

dict_keys(['Resume_data.zip'])


In [21]:
import zipfile
import os

zip_path = "Resume_data.zip"
extract_dir = "resumes_pdf"

with zipfile.ZipFile(zip_path, "r") as zip_ref:
    zip_ref.extractall(extract_dir)

data_dir = os.path.join(extract_dir, "data")

print("Number of categories:", len(os.listdir(data_dir)))
print(os.listdir(data_dir))

Number of categories: 24
['DIGITAL-MEDIA', 'AVIATION', 'SALES', 'ADVOCATE', 'ARTS', 'ACCOUNTANT', 'TEACHER', 'HEALTHCARE', 'ENGINEERING', 'BPO', 'BANKING', 'APPAREL', 'PUBLIC-RELATIONS', 'CONSULTANT', 'FITNESS', 'INFORMATION-TECHNOLOGY', 'DESIGNER', 'FINANCE', 'CHEF', 'AGRICULTURE', 'AUTOMOBILE', 'BUSINESS-DEVELOPMENT', 'CONSTRUCTION', 'HR']


In [22]:
!pip install -q pymupdf

In [23]:
import pymupdf
import pandas as pd

In [26]:
extracted_resumes = []

for folder in os.listdir(data_dir):

    folder_path = os.path.join(data_dir, folder)

    if not os.path.isdir(folder_path):
        continue

    for file in os.listdir(folder_path):

        if not file.lower().endswith(".pdf"):
            continue

        file_path = os.path.join(folder_path, file)

        try:
            doc = pymupdf.open(file_path)

            text = ""

            for page in doc:
                text += page.get_text()

            doc.close()

            extracted_resumes.append({
                "filename": file,
                "label": folder,
                "text": text
            })

        except Exception as e:
            print(f"Error reading {file_path}: {e}")

In [27]:
df = pd.DataFrame(extracted_resumes)

print("Total resumes:", len(df))
print("Shape:", df.shape)

df.head()

Total resumes: 2484
Shape: (2484, 3)


,filename,label,text
0,24953921.pdf,DIGITAL-MEDIA,MEDIA SERVICES COORDINATOR\nSummary\nLife-long...
1,13837784.pdf,DIGITAL-MEDIA,DIGITAL MEDIA BUYER\nProfessional Summary\nVer...
2,18354623.pdf,DIGITAL-MEDIA,DIGITAL MARKETING MANAGER\nCareer Focus\nDigit...
3,40311088.pdf,DIGITAL-MEDIA,MEDIA SPECIALIST II\nProfessional Summary\nI w...
4,16893572.pdf,DIGITAL-MEDIA,DIGITAL MARKETING MANAGER\nSummary\nCreative m...


In [28]:
df["text"] = df["text"].fillna("").str.strip()

df = df[df["text"] != ""].reset_index(drop=True)

print("Usable resumes:", len(df))

Usable resumes: 2483


In [29]:
!pip install -q sentence-transformers faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 34.2 MB/s eta 0:00:00


In [30]:
from sentence_transformers import SentenceTransformer
import faiss
import numpy as np

In [31]:
embedding_model = SentenceTransformer(
    "all-MiniLM-L6-v2"
)

print("Model loaded!")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Model loaded!


In [32]:
sample_embedding = embedding_model.encode(
    df.loc[0, "text"],
    convert_to_numpy=True
)

print(sample_embedding.shape)

(384,)


In [33]:
resume_embeddings = embedding_model.encode(
    df["text"].tolist(),
    convert_to_numpy=True,
    show_progress_bar=True,
    batch_size=32
)

print(resume_embeddings.shape)

Batches:   0%|          | 0/78 [00:00<?, ?it/s]

(2483, 384)


In [34]:
resume_embeddings = resume_embeddings.astype("float32")

faiss.normalize_L2(resume_embeddings)

In [35]:
embedding_dimension = resume_embeddings.shape[1]

index = faiss.IndexFlatIP(
    embedding_dimension
)

index.add(resume_embeddings)

print("Number of resumes in index:", index.ntotal)

Number of resumes in index: 2483


In [36]:
def search_resumes(query, top_k=5):

    query_embedding = embedding_model.encode(
        [query],
        convert_to_numpy=True
    ).astype("float32")

    faiss.normalize_L2(query_embedding)

    scores, indices = index.search(
        query_embedding,
        top_k
    )

    results = df.iloc[indices[0]].copy()

    results["similarity_score"] = scores[0]

    return results[
        [
            "filename",
            "label",
            "similarity_score",
            "text"
        ]
    ]

In [37]:
query = """
Candidate with experience in Python, machine learning,
data analysis and predictive modeling.
"""

results = search_resumes(query, top_k=5)

results[
    ["filename", "label", "similarity_score"]
]

,filename,label,similarity_score
2121,18448085.pdf,AUTOMOBILE,0.528977
1003,41152404.pdf,BPO,0.498680
1666,37242217.pdf,INFORMATION-TECHNOLOGY,0.490032
1232,10562768.pdf,APPAREL,0.483915
909,12011623.pdf,ENGINEERING,0.478226


In [38]:
for i, row in results.iterrows():

    print("=" * 80)

    print("Filename:", row["filename"])
    print("Category:", row["label"])
    print("Similarity:", round(row["similarity_score"], 4))

    print("\nResume preview:")
    print(row["text"][:1000])

Filename: 18448085.pdf
Category: AUTOMOBILE
Similarity: 0.529

Resume preview:
DATA ANALYST
Professional Summary
Industrial and Systems Engineering graduate, certified Base SAS Programmer and a Lean Six Sigma Green Belt with strong background in
statistics, mathematics and logical problem solving looking for a dynamic opportunity in data driven fields of analytics and statistical modeling.
Core Qualifications
Data Science Tools: R, Base SAS, Python (Numpy, Pandas, Matplotlib, Scikit- learn), SPSS, Minitab, MATLAB, Apache Spark, SQL, MS
Excel, MS Visio, Tableau MySQL, Oracle Database, Microsoft Access Key Competencies: Data Extraction, Data Wrangling, Data Analysis,
Data Visualization, Regression Analysis (Linear, Logistic and Multinomial), Time Series Analysis, Association Rule Mining, Monte Carlo
Simulation, Optimization, Random Forests
Experience
07/2016 to Current
Data Analyst Company Name ï¼​ State
09/2015 to 05/2016
Student Manager Company Name ï¼​ State
Undertook a leadership and

In [39]:
test_queries = {
    "Machine Learning": """
    Python, machine learning, data analysis, predictive modeling,
    statistics, deep learning and data science
    """,

    "Human Resources": """
    Human resources, recruitment, employee relations, onboarding,
    HR administration, payroll and personnel management
    """,

    "Finance": """
    Financial analysis, accounting, auditing, budgeting,
    financial reporting, Excel and financial management
    """,

    "Software / IT": """
    Software development, Python, Java, programming,
    databases, web development and technical systems
    """
}

In [40]:
all_search_results = {}

for query_name, query_text in test_queries.items():

    print("\n" + "=" * 100)
    print("QUERY:", query_name)
    print("=" * 100)

    results = search_resumes(
        query_text,
        top_k=5
    )

    all_search_results[query_name] = results

    print(
        results[
            ["filename", "label", "similarity_score"]
        ].to_string(index=False)
    )


QUERY: Machine Learning
    filename                  label  similarity_score
18448085.pdf             AUTOMOBILE          0.425257
12011623.pdf            ENGINEERING          0.394623
22946204.pdf             AUTOMOBILE          0.391666
62994611.pdf            AGRICULTURE          0.376799
20674668.pdf INFORMATION-TECHNOLOGY          0.369640

QUERY: Human Resources
    filename   label  similarity_score
32947778.pdf      HR          0.670653
87968870.pdf      HR          0.663948
23011221.pdf FITNESS          0.662030
14225422.pdf      HR          0.658777
17812897.pdf      HR          0.657158

QUERY: Finance
    filename      label  similarity_score
24833063.pdf    FINANCE          0.638607
12802330.pdf ACCOUNTANT          0.600305
31948488.pdf    FINANCE          0.595494
27637576.pdf ACCOUNTANT          0.592054
81677620.pdf    FINANCE          0.589590

QUERY: Software / IT
    filename                  label  similarity_score
62994611.pdf            AGRICULTURE          0.55

In [41]:
for query_name, results in all_search_results.items():

    print("\n\n" + "#" * 100)
    print("QUERY:", query_name)
    print("#" * 100)

    for rank, (_, row) in enumerate(
        results.head(3).iterrows(),
        start=1
    ):

        print("\n" + "-" * 80)
        print(f"RANK {rank}")
        print("Filename:", row["filename"])
        print("Category:", row["label"])
        print("Similarity:", round(row["similarity_score"], 4))

        print("\nResume Preview:")
        print(row["text"][:1200])



####################################################################################################
QUERY: Machine Learning
####################################################################################################

--------------------------------------------------------------------------------
RANK 1
Filename: 18448085.pdf
Category: AUTOMOBILE
Similarity: 0.4253

Resume Preview:
DATA ANALYST
Professional Summary
Industrial and Systems Engineering graduate, certified Base SAS Programmer and a Lean Six Sigma Green Belt with strong background in
statistics, mathematics and logical problem solving looking for a dynamic opportunity in data driven fields of analytics and statistical modeling.
Core Qualifications
Data Science Tools: R, Base SAS, Python (Numpy, Pandas, Matplotlib, Scikit- learn), SPSS, Minitab, MATLAB, Apache Spark, SQL, MS
Excel, MS Visio, Tableau MySQL, Oracle Database, Microsoft Access Key Competencies: Data Extraction, Data Wrangling, Data Analysis,
Data Vis

In [42]:
evaluation_records = []

for query_name, results in all_search_results.items():

    for rank, (_, row) in enumerate(
        results.iterrows(),
        start=1
    ):

        evaluation_records.append({
            "query": query_name,
            "rank": rank,
            "filename": row["filename"],
            "label": row["label"],
            "similarity_score": row["similarity_score"]
        })

evaluation_df = pd.DataFrame(evaluation_records)

evaluation_df

,query,rank,filename,label,similarity_score
0,Machine Learning,1,18448085.pdf,AUTOMOBILE,0.425257
1,Machine Learning,2,12011623.pdf,ENGINEERING,0.394623
2,Machine Learning,3,22946204.pdf,AUTOMOBILE,0.391666
3,Machine Learning,4,62994611.pdf,AGRICULTURE,0.376799
4,Machine Learning,5,20674668.pdf,INFORMATION-TECHNOLOGY,0.369640
5,Human Resources,1,32947778.pdf,HR,0.670653
6,Human Resources,2,87968870.pdf,HR,0.663948
7,Human Resources,3,23011221.pdf,FITNESS,0.662030
8,Human Resources,4,14225422.pdf,HR,0.658777
9,Human Resources,5,17812897.pdf,HR,0.657158


In [43]:
evaluation_df["relevant"] = None

evaluation_df.head()

,query,rank,filename,label,similarity_score,relevant
0,Machine Learning,1,18448085.pdf,AUTOMOBILE,0.425257,None
1,Machine Learning,2,12011623.pdf,ENGINEERING,0.394623,None
2,Machine Learning,3,22946204.pdf,AUTOMOBILE,0.391666,None
3,Machine Learning,4,62994611.pdf,AGRICULTURE,0.376799,None
4,Machine Learning,5,20674668.pdf,INFORMATION-TECHNOLOGY,0.369640,None


In [44]:
!pip install -q groq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.7/143.7 kB 2.1 MB/s eta 0:00:00


In [45]:
from groq import Groq

In [ ]:
# API key is loaded securely using an environment variable.
# Set GROQ_API_KEY before running the LLM section.

In [47]:
client = Groq(
    api_key=os.environ.get("GROQ_API_KEY")
)

In [48]:
completion = client.chat.completions.create(
    model="llama-3.1-8b-instant",
    messages=[
        {
            "role": "user",
            "content": "Say hello and confirm that you are working."
        }
    ]
)

print(completion.choices[0].message.content)

Hello. I'm functioning as intended and I'm here to assist you with any questions or topics you'd like to discuss.


In [49]:
def retrieve_context(query, top_k=3):

    results = search_resumes(query, top_k=top_k)

    context = ""

    for rank, (_, row) in enumerate(results.iterrows(), start=1):

        context += f"""

RESUME {rank}
Filename: {row['filename']}
Category: {row['label']}

{row['text'][:2500]}

----------------------------
"""

    return context, results

In [50]:
query = """
Find candidates with experience in Python, machine learning,
data analysis and predictive modeling.
"""

context, retrieved_results = retrieve_context(
    query,
    top_k=3
)

print(
    retrieved_results[
        ["filename", "label", "similarity_score"]
    ]
)

print("\nCONTEXT PREVIEW:\n")
print(context[:3000])

          filename        label  similarity_score
2121  18448085.pdf   AUTOMOBILE          0.517544
909   12011623.pdf  ENGINEERING          0.493517
2077  62994611.pdf  AGRICULTURE          0.490451

CONTEXT PREVIEW:


        
RESUME 1
Filename: 18448085.pdf
Category: AUTOMOBILE

DATA ANALYST
Professional Summary
Industrial and Systems Engineering graduate, certified Base SAS Programmer and a Lean Six Sigma Green Belt with strong background in
statistics, mathematics and logical problem solving looking for a dynamic opportunity in data driven fields of analytics and statistical modeling.
Core Qualifications
Data Science Tools: R, Base SAS, Python (Numpy, Pandas, Matplotlib, Scikit- learn), SPSS, Minitab, MATLAB, Apache Spark, SQL, MS
Excel, MS Visio, Tableau MySQL, Oracle Database, Microsoft Access Key Competencies: Data Extraction, Data Wrangling, Data Analysis,
Data Visualization, Regression Analysis (Linear, Logistic and Multinomial), Time Series Analysis, Association Rule Mining,

In [51]:
def ask_resume_rag(question, top_k=3):

    context, results = retrieve_context(
        question,
        top_k=top_k
    )

    messages = [
        {
            "role": "system",
            "content": """
You are a recruitment assistant analyzing retrieved resumes.

Answer the user's question ONLY using the information in the
provided resume context.

Compare candidates when the question asks which candidate is
strongest or most suitable.

Use evidence from the resumes, such as skills, tools, projects,
experience, and qualifications, to make a reasoned comparison.

Do not say that you cannot determine the answer if the resume
context contains enough evidence to make a reasonable comparison.
If evidence is limited, state that clearly but still provide the
best-supported conclusion.
"""
        },
        {
            "role": "user",
            "content": f"""
RESUME CONTEXT:

{context}

QUESTION:
{question}
"""
        }
    ]

    completion = client.chat.completions.create(
        model="llama-3.1-8b-instant",
        messages=messages,
        temperature=0.2
    )

    answer = completion.choices[0].message.content

    return answer, results

In [52]:
question = """
Which candidate appears to have the strongest experience in
machine learning and predictive modeling? Explain why.
"""

answer, rag_results = ask_resume_rag(
    question,
    top_k=3
)

print(answer)

Based on the provided resume context, Resume 3 (PHD CANDIDATE IN FINANCE) appears to have the strongest experience in machine learning and predictive modeling.

The candidate has listed proficiency in R, SAS, SQL, C++, Fortran, and VBA, which are programming languages commonly used in machine learning and predictive modeling. Additionally, they have highlighted advanced modeling skills, project management experience, critical thinking skills, complex problem-solving skills, and superior research skills.

Their research experience, as a PhD candidate in finance, includes conducting research in finance, specifically asset pricing and econometric modeling, using advanced statistical techniques such as Maximum Entropy Econometrics and GARCH Process. They have also worked on projects related to market sentiment and its effects on stock returns, which involves predictive modeling.

While Resume 2 (VP, PRINCIPAL) has experience in designing, developing, and implementing mission-critical syste

In [53]:
question = """
Which retrieved candidate has the strongest technical and programming
skill set? Compare the candidates and explain your reasoning.
"""

answer, rag_results = ask_resume_rag(question, top_k=3)

print(answer)

Based on the provided resume context, I would conclude that Resume 2 (Information Designer) has the strongest technical and programming skill set.

Here's a comparison of the candidates:

- Resume 1 (IT Consultant) has a broad range of technical skills, including proficiency in Microsoft Office, various IT management tools, and experience with Windows, Linux, and Unix operating systems. However, the skills listed are more focused on IT management and administration rather than programming.

- Resume 2 (Information Designer) lists a wide range of programming languages, including C, C++, C#.NET, Java, Python, PHP, and Mathematica. The candidate also has experience with various software development tools, such as TOAD for Oracle, SQL Developer, Visual Studio, and Eclipse. Additionally, the candidate has experience with Unix/Linux System Administration, Oracle Database Administration, and MySQL database administration.

- Resume 3 (Information Technology Consultant) lists a broad range of 

In [54]:
question = """
Summarize the key skills and experience of the most relevant
candidate for machine learning and data analysis.
"""

answer, rag_results = ask_resume_rag(question, top_k=3)

print(answer)

Based on the provided resume context, the most relevant candidate for machine learning and data analysis is Resume 1 (Engineering and Quality Technician) and Resume 2 (Data Analyst).

However, Resume 1 has more extensive experience in machine learning and data analysis, with skills in:

* Predictive modeling
* Data visualization
* SQL
* Web scraping
* R, SAS, and Python programming
* Experience in handling large datasets and data warehousing
* Familiarity with various machine learning algorithms, including Random Forests, Decision Trees, and Boosted Trees

Resume 2 also has strong skills in machine learning and data analysis, with experience in:

* Data Science Tools: R, Base SAS, Python (Numpy, Pandas, Matplotlib, Scikit-learn), SPSS, Minitab, MATLAB, Apache Spark, SQL, MS Excel, MS Visio, Tableau MySQL, Oracle Database, Microsoft Access
* Key Competencies: Data Extraction, Data Wrangling, Data Analysis, Data Visualization, Regression Analysis (Linear, Logistic and Multinomial), Time 

In [55]:
question = """
What machine learning, statistical modeling, and data analysis
techniques are mentioned across the retrieved resumes?
"""

answer, rag_results = ask_resume_rag(question, top_k=3)

print(answer)

Based on the provided resumes, the following machine learning, statistical modeling, and data analysis techniques are mentioned:

1. **Machine Learning:**
	* Random Forests (RESUME 2, RESUME 3)
	* Decision Trees (RESUME 2)
	* Boosted Trees (RESUME 2)
	* Ridge, LASSO, and Elastic Net regression models (RESUME 2)
	* Logistic Regression (RESUME 2, RESUME 3)
	* Monte Carlo Simulation (RESUME 3)
2. **Statistical Modeling:**
	* Predictive modeling (RESUME 1)
	* Data visualization (RESUME 1, RESUME 2, RESUME 3)
	* Statistical analysis (RESUME 2, RESUME 3)
	* Regression Analysis (Linear, Logistic, and Multinomial) (RESUME 3)
	* Time Series Analysis (RESUME 3)
	* Association Rule Mining (RESUME 3)
3. **Data Analysis:**
	* Data extraction (RESUME 3)
	* Data wrangling (RESUME 3)
	* Data analysis (RESUME 2, RESUME 3)
	* Outlier analysis (RESUME 2)
	* Dimension Reduction using PCA (RESUME 2)
	* Clustering techniques (RESUME 2)
	* Conjoint analysis (RESUME 2)
	* Exploratory factor analysis (RESUME 3

In [59]:
evaluation_df["relevant"] = 0
evaluation_df

,query,rank,filename,label,similarity_score,relevant
0,Machine Learning,1,18448085.pdf,AUTOMOBILE,0.425257,0
1,Machine Learning,2,12011623.pdf,ENGINEERING,0.394623,0
2,Machine Learning,3,22946204.pdf,AUTOMOBILE,0.391666,0
3,Machine Learning,4,62994611.pdf,AGRICULTURE,0.376799,0
4,Machine Learning,5,20674668.pdf,INFORMATION-TECHNOLOGY,0.369640,0
5,Human Resources,1,32947778.pdf,HR,0.670653,0
6,Human Resources,2,87968870.pdf,HR,0.663948,0
7,Human Resources,3,23011221.pdf,FITNESS,0.662030,0
8,Human Resources,4,14225422.pdf,HR,0.658777,0
9,Human Resources,5,17812897.pdf,HR,0.657158,0


In [60]:
evaluation_df.loc[
    evaluation_df["query"] == "Machine Learning",
    "relevant"
] = [1, 1, 1, 1, 0]

evaluation_df.loc[
    evaluation_df["query"] == "Human Resources",
    "relevant"
] = [1, 1, 1, 1, 1]


evaluation_df.loc[
    evaluation_df["query"] == "Finance",
    "relevant"
] = [1, 1, 1, 1, 1]


evaluation_df.loc[
    evaluation_df["query"] == "Software / IT",
    "relevant"
] = [1, 1, 1, 1, 1]

In [61]:
evaluation_df[
    ["query", "rank", "filename", "relevant"]
]

,query,rank,filename,relevant
0,Machine Learning,1,18448085.pdf,1
1,Machine Learning,2,12011623.pdf,1
2,Machine Learning,3,22946204.pdf,1
3,Machine Learning,4,62994611.pdf,1
4,Machine Learning,5,20674668.pdf,0
5,Human Resources,1,32947778.pdf,1
6,Human Resources,2,87968870.pdf,1
7,Human Resources,3,23011221.pdf,1
8,Human Resources,4,14225422.pdf,1
9,Human Resources,5,17812897.pdf,1


In [62]:
precision_at_5 = (
    evaluation_df
    .groupby("query")["relevant"]
    .mean()
    .reset_index()
)

precision_at_5.columns = [
    "query",
    "Precision@5"
]

precision_at_5

,query,Precision@5
0,Finance,1.0
1,Human Resources,1.0
2,Machine Learning,0.8
3,Software / IT,1.0


In [63]:
mean_precision_at_5 = precision_at_5["Precision@5"].mean()

print(
    f"Mean Precision@5: {mean_precision_at_5:.3f}"
)

Mean Precision@5: 0.950


In [64]:
evaluation_df.to_csv(
    "retrieval_evaluation.csv",
    index=False
)

precision_at_5.to_csv(
    "precision_at_5_results.csv",
    index=False
)

print("Evaluation results saved!")

Evaluation results saved!


In [65]:
faiss.write_index(index, "resume_faiss.index")

df[["filename", "label", "text"]].to_pickle(
    "resume_metadata.pkl"
)

print("FAISS index and metadata saved successfully!")

FAISS index and metadata saved successfully!


In [66]:
from google.colab import files

files.download("resume_faiss.index")
files.download("resume_metadata.pkl")
files.download("retrieval_evaluation.csv")
files.download("precision_at_5_results.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>